# MLIR → SPIR-V → WGSL → WebGPU

Write a GPU kernel in Python, compile it in this tab, and run it on your GPU.

Every step happens client-side. There is no server and nothing is precompiled:
MLIR, its pass pipeline, and the SPIR-V→WGSL translator (Tint) are all compiled
to WebAssembly and shipped in the `mlir-python-bindings` wheel. The dispatch goes
straight to the browser's own WebGPU implementation.

The kernel is authored with `eudsl-python-extras`, so there is no MLIR text
anywhere below — the IR is built through the Python bindings.

**Requires a WebGPU-capable browser.** Recent Chrome and Edge work; Safari and
Firefox need it enabled.

In [ ]:
%%capture

import piplite

# numpy FIRST, and this order matters. mlir-python-bindings declares
# `Requires-Dist: numpy`, so installing it first makes micropip resolve two
# wheels and install them concurrently (micropip/install.py does an
# asyncio.gather over them). The dynamic libraries then load interleaved, and
# because 20 side modules in the wheel each carry their own copy of LLVM's
# cl::opt registry, whichever registers second aborts the whole module:
#     LLVM ERROR: inconsistency in registered CommandLine options
# which surfaces here as `JsException: RuntimeError: Aborted()`. Satisfying the
# dependency up front keeps it to one wheel at a time. See llvm/eudsl#502.
await piplite.install('numpy')
await piplite.install('mlir-python-bindings')
await piplite.install('eudsl-python-extras')

In [ ]:
from mlir.webgpu import get_device

# Raises with a clear message if this browser has no WebGPU.
device = await get_device()
print("WebGPU adapter acquired")

## 1. The kernel

A 32×32 matmul, written with the `eudsl-python-extras` GPU dialect helpers:
`@gpu.func` builds a `gpu.func`, `block_dim`/`block_idx`/`thread_idx` become the
corresponding `gpu` ops, and `scf.range_` becomes an `scf.for` carrying the
accumulator as an iteration argument.

Two attributes matter for what follows:

- `spirv.entry_point_abi` carries the workgroup size, which becomes
  `@workgroup_size` in the generated WGSL.
- `TARGET_ENV`, attached to the `gpu.module`, uses the `Shader` capability,
  giving a `Logical`/`GLSL450` module. That is the Vulkan flavour of
  SPIR-V, which is what WebGPU accepts — the `Kernel`/OpenCL flavour used for CPU
  and Level Zero targets will not translate.

In [ ]:
import mlir.extras.types as T
from mlir.extras.context import RAIIMLIRContextModule
# memref is imported for its side effect: it registers a value caster
# (memref.py, register_value_caster) that gives memref-typed block
# arguments __getitem__, which is what makes A[row, k] work below.
from mlir.extras.dialects import arith, gpu, memref, scf
from mlir.extras.dialects.gpu import (
    block_dim,
    block_idx,
    module,
    set_container_module,
    thread_idx,
)
from mlir.ir import Attribute

ctx = RAIIMLIRContextModule()
set_container_module(ctx.module)

M = N = K = 32
WG = 8
dtype = T.f32()

WORKGROUP_SIZE = Attribute.parse(
    f"#spirv.entry_point_abi<workgroup_size = [{WG}, {WG}, 1]>"
)

# Vulkan flavour: the Shader capability gives a Logical/GLSL450 module, which is
# what WebGPU accepts. SPV_KHR_storage_buffer_storage_class is not optional --
# convert-gpu-to-spirv maps the memref arguments to StorageBuffer, and without
# the extension named here `memref.load` fails to legalize.
TARGET_ENV = (
    "#spirv.target_env<#spirv.vce<v1.0, [Shader],"
    " [SPV_KHR_storage_buffer_storage_class]>, api=Vulkan,"
    " #spirv.resource_limits<>>"
)


@gpu.func(func_attrs={"spirv.entry_point_abi": WORKGROUP_SIZE})
def matmul(
    A: T.memref(M, K, dtype), B: T.memref(K, N, dtype), C: T.memref(M, N, dtype)
):
    row = block_dim.x * block_idx.x + thread_idx.x
    col = block_dim.y * block_idx.y + thread_idx.y

    acc = arith.constant(0.0, type=dtype)
    for k, acc, _ in scf.range_(K, iter_args=[acc]):
        acc += A[row, k] * B[k, col]
        acc = scf.yield_(acc)

    C[row, col] = acc


@module("kernels", [TARGET_ENV])
def kernels():
    matmul.emit()


print(ctx.module)

## 2. Lower to the SPIR-V dialect

`convert-gpu-to-spirv` rewrites the kernel body. Then `spirv-lower-abi-attrs`
turns the `memref` arguments into `spirv.GlobalVariable`s with `bind(set,
binding)` decorations — Vulkan requires entry points to be `void(void)`, so
arguments become interface variables. `spirv-update-vce` computes the
`vce_triple` the serializer needs, and `spirv-webgpu-prepare` expands the ops
WebGPU does not allow.

The pipeline is built with `Pipeline()` rather than a pass-pipeline string.

In [ ]:
from mlir.extras.runtime.passes import Pipeline, run_pipeline

# No spirv-attach-target here: the gpu.module already carries TARGET_ENV, and
# that pass does not replace a target that is already present.
lower = (
    Pipeline()
    .convert_gpu_to_spirv()
    .Spirv(
        Pipeline()
        .spirv_lower_abi_attrs()
        .spirv_update_vce()
        .spirv_webgpu_prepare()
    )
)
print(lower, end="\n\n")

lowered = run_pipeline(ctx.module, lower)

asm = str(lowered)
print(asm[asm.index("spirv.module") : asm.index("gpu.module")])

Note the `bind(0, 0)`, `bind(0, 1)`, `bind(0, 2)` on the global variables. Those
become `@group(0) @binding(N)` in WGSL and decide how the host binds buffers, in
kernel-argument order.

## 3. Serialize to a SPIR-V binary

`gpu-module-to-binary` calls `spirv::serialize` internally and stores the result
in a `gpu.binary` op, which `get_compile_object_bytes` reads back out.

One wart, handled by `mlir.webgpu.nest_spirv_module`: `convert-gpu-to-spirv`
hoists the generated `spirv.module` to the top level as a *sibling* of the
emptied `gpu.module`, while the serializer only looks *inside* one. Upstream's
Vulkan runner pipeline avoids this with
`test-convert-to-spirv{nest-in-gpu-module=true}`, but that pass is test-only and
is not in this wheel.

In [ ]:
from mlir.extras.dialects.gpu import get_compile_object_bytes
from mlir.ir import Module
from mlir.webgpu import nest_spirv_module

nested = Module.parse(nest_spirv_module(str(lowered)))
binary = run_pipeline(nested, Pipeline().gpu_module_to_binary())

spirv = bytes(get_compile_object_bytes(binary))
magic = int.from_bytes(spirv[:4], "little")
print(f"{len(spirv)} bytes, magic {magic:#010x} (SPIR-V is 0x07230203)")

## 4. SPIR-V → WGSL

`mlir.wgsl` wraps Tint's SPIR-V reader and WGSL writer, compiled into the wheel.
WebGPU only accepts WGSL, so this step is unavoidable — but note it is *only* the
translator. Dawn's runtime is not here; the browser already implements WebGPU.

In [ ]:
from mlir.wgsl import spirv_to_wgsl

wgsl = spirv_to_wgsl(spirv)
print(wgsl)

## 5. Dispatch

`mlir.webgpu.dispatch` uploads the inputs, runs the shader, and reads the result
back. Buffers bind at `@group(0) @binding(i)` — inputs in order, output last —
which lines up with the `bind(0, N)` decorations from step 2 without either side
being adjusted to fit the other.

In [ ]:
import numpy as np
from mlir.webgpu import dispatch

rng = np.random.default_rng(0)
A = rng.standard_normal((M, K), dtype=np.float32)
B = rng.standard_normal((K, N), dtype=np.float32)

C = await dispatch(
    wgsl,
    inputs=[A, B],
    out_shape=(M, N),
    entry_point="matmul",
    workgroup=(WG, WG, 1),
    device=device,
)

expected = A @ B
print("max abs err:", float(np.max(np.abs(C - expected))))
print("MATCH" if np.allclose(C, expected, rtol=1e-4, atol=1e-4) else "MISMATCH")

The error is float32 accumulation noise over a K=32 dot product, not a
correctness problem.

## Try your own

Edit the `@gpu.func` in step 1 and re-run from there. Things worth knowing:

- Argument types come from `mlir.extras.types`; `T.memref(...)` of `f32` or `i32`
  is safe. WebGPU has no `f64`.
- Keep the capability set to `Shader`. Adding capabilities the browser lacks makes
  Tint reject the module, usually with a message about an unsupported SPIR-V
  feature.
- Buffers bind at `@group(0) @binding(i)` in argument order, so keep the output
  last if you use `dispatch` as-is.
- If `spirv_to_wgsl` raises, the message carries Tint's own diagnostics plus a
  SPIR-V header summary, which is normally enough to see which op it choked on.
- `run_pipeline` prints a reproducer if a pass fails, including the pipeline and
  the offending module.